# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [ ]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


Note: you may need to restart the kernel to use updated packages.


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [ ]:
import pandas as pd
from datasets import Dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [ ]:
DATA_PATH  = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df         = pd.read_parquet(DATA_PATH)
hf_dataset = Dataset.from_pandas(df)

TRAIN_START = 0
TRAIN_END   = 4000
VAL_START   = 4000
VAL_END     = 5000

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds   = hf_dataset.select(range(VAL_START, VAL_END))

display(df.head())
print(f"Train size : {len(train_ds)} | Val size : {len(val_ds)}")
print("Colonnes   :", df.columns.tolist())
print("Spam rate  :", df['label'].mean().round(3))

# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [ ]:
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded : {MODEL_NAME}")
print(f"Pad token        : {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})")

In [ ]:
TEXT_COLUMN      = 'sms'
PADDING_STRATEGY = 'max_length'
TRUNCATION_FLAG  = True
MAX_SEQ_LEN      = 64


def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )


train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn, batched=True)

print(f"Tokenization done. Train : {len(train_tok)} | Val : {len(val_tok)}")
print("Colonnes :", train_tok.column_names)

# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [ ]:
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,
)

print(f"GPT-2 classifier loaded with {NUM_LABELS} labels.")
print(f"Device : {'GPU' if torch.cuda.is_available() else 'CPU'}")

# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5  # 1/sqrt(d_k) pour stabiliser les gradients

    def forward(self, query, key, value, mask=None):
        # scores : (batch, seq, seq)
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, value), attn


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn      = Attention(embed_dim)
        self.fc        = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed             = self.embedding(x)               # (batch, seq, embed)
        attn_output, _    = self.attn(embed, embed, embed)  # self-attention : q=k=v
        pooled            = attn_output.mean(dim=1)         # mean-pool sur la dimension seq
        return self.fc(pooled)


print("Attention et SimpleAttentionClassifier définis.")

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [ ]:
ATTN_TEXT_COLUMN = 'sms'
ATTN_MAX_LEN     = 64


def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}


train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn   = val_ds.map(preprocess_for_attention)

print(f"Preprocessing done. Train : {len(train_ds_attn)} | Val : {len(val_ds_attn)}")

In [ ]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label':     torch.tensor(item['label'],     dtype=torch.long),
        }


TRAIN_DATA_FOR_LOADER = train_ds_attn
VAL_DATA_FOR_LOADER   = val_ds_attn

train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader   = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER),   batch_size=32)

print(f"Train batches : {len(train_loader)} | Val batches : {len(val_loader)}")

In [ ]:
vocab_size    = tokenizer.vocab_size + len(tokenizer.get_added_vocab())
embed_dim     = 64
num_classes   = 2
learning_rate = 1e-3

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer  = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion  = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss    = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', round(loss.item(), 4))

# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [ ]:
import evaluate
import numpy as np

accuracy  = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall    = evaluate.load('recall')
f1        = evaluate.load('f1')


def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy.compute( predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall':    recall.compute(   predictions=preds, references=labels)['recall'],
        'f1':        f1.compute(       predictions=preds, references=labels)['f1'],
    }


print("Métriques chargées : accuracy, precision, recall, f1")

> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [ ]:
gpt2_preds  = []
gpt2_labels = []
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    'accuracy':  accuracy.compute( predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall':    recall.compute(   predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1':        f1.compute(       predictions=gpt2_preds, references=gpt2_labels)['f1'],
}
print('GPT-2 Metrics:', gpt2_metrics)

In [ ]:
attn_preds  = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds   = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    'accuracy':  accuracy.compute( predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall':    recall.compute(   predictions=attn_preds, references=attn_labels)['recall'],
    'f1':        f1.compute(       predictions=attn_preds, references=attn_labels)['f1'],
}
print('Attention Model Metrics:', attn_metrics)

# Comparaison côte à côte
import pandas as pd
comparison = pd.DataFrame([gpt2_metrics, attn_metrics], index=['GPT-2 (pretrained)', 'Custom Attention'])
print("\n=== Comparaison des deux modèles ===")
display(comparison.round(4))

# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?

Dans le mécanisme d'attention, chaque token produit trois vecteurs : **Query (Q)**, **Key (K)** et **Value (V)**.

- **Query (Q)** : représente la "question" que pose un token — ce qu'il cherche dans le contexte. C'est le token qui veut s'informer.
- **Key (K)** : représente l'"étiquette" de chaque token — ce qu'il peut offrir comme information aux autres tokens.
- **Value (V)** : contient le contenu réel transporté par chaque token, qui sera récupéré si l'attention est forte.

**Fonctionnement :** on calcule la similarité entre le Query d'un token et les Keys de tous les autres tokens (`QKᵀ`). Plus le score est élevé, plus le token correspondant "attire" l'attention. Ces scores (après softmax) pondèrent les Values, ce qui produit une représentation enrichie du contexte.

**Analogie :** le Query est une requête de recherche, les Keys sont les titres des documents, et les Values sont le contenu des documents retournés.

### 2. Why do we use a scaling factor in the dot-product attention?

Le produit scalaire `QKᵀ` produit des valeurs dont la magnitude croît avec la dimension `d_k` des embeddings. Avec de grandes valeurs, la fonction **softmax** sature : elle produit des probabilités très proches de 0 ou 1, ce qui écrase les gradients et bloque l'apprentissage.

En divisant par `√d_k`, on ramène les scores dans une plage où la softmax reste "douce" (non saturée), ce qui garantit :
- Des **gradients stables** lors de la rétropropagation
- Une **distribution d'attention diffuse** au début de l'entraînement (le modèle peut apprendre quoi regarder progressivement)
- Un comportement **indépendant de la dimension** des embeddings

Sans ce facteur d'échelle, augmenter la taille des embeddings dégraderait automatiquement la qualité de l'entraînement.

### 3. How does self-attention differ from traditional sequence models like RNNs?

| Critère | RNN / LSTM | Self-Attention |
|---|---|---|
| **Traitement** | Séquentiel (token par token) | Parallèle (tous les tokens en même temps) |
| **Dépendances longues** | Difficile : l'information doit "traverser" de nombreuses étapes | Direct : chaque token accède à tous les autres en une seule opération |
| **Mémoire** | Encodée dans un vecteur caché `h_t` qui s'écrase progressivement | Distribuée sur toute la séquence via les scores d'attention |
| **Complexité temporelle** | O(n) — linéaire en longueur de séquence | O(n²) — quadratique (chaque token × chaque token) |
| **Entraînement** | Lent à paralléliser (dépendance temporelle) | Très parallélisable → idéal sur GPU |
| **Vanishing gradient** | Problème connu pour les longues séquences | Absent : les chemins entre tokens sont directs |

**En résumé :** les RNNs traitent les séquences comme une chaîne (un maillon à la fois), ce qui limite leur capacité à relier des tokens distants. Le self-attention traite la séquence comme un graphe complet où chaque token peut directement interagir avec n'importe quel autre token, ce qui le rend beaucoup plus puissant pour capturer le contexte global.

### 4. Performance analysis

#### Quel modèle performe le mieux ?

**GPT-2 (pretrained)** surpasse très probablement le modèle custom, même sans fine-tuning, grâce à ses 117M de paramètres préentraînés sur des milliards de tokens. Sa représentation interne du langage est riche et généraliste.

**Le modèle Custom Attention** est beaucoup plus léger (embedding 64 dimensions + une couche d'attention), et n'est entraîné que sur 4 000 exemples pendant une seule epoch. Il peut tout de même atteindre ~85-90% d'accuracy sur ce dataset déséquilibré (86% ham / 14% spam) simplement en prédisant toujours la classe majoritaire — donc les métriques `precision`, `recall` et `f1` sur la classe spam sont les vraies mesures discriminantes.

#### Trade-offs

| Aspect | GPT-2 | Custom Attention |
|---|---|---|
| **Performance** | Élevée (représentations riches) | Modérée (sous-paramétrisé) |
| **Vitesse d'inférence** | Lente (117M params) | Très rapide (quelques milliers de params) |
| **Ressources** | GPU recommandé, 500MB+ | CPU suffit, <1MB |
| **Interprétabilité** | Difficile (boîte noire) | Facile à inspecter (scores d'attention) |
| **Fine-tuning nécessaire** | Non (zero-shot) ou oui | Oui, mais rapide |

#### Amélioration suggérée pour le Custom Attention Classifier

Ajouter **plusieurs têtes d'attention (Multi-Head Attention)** : au lieu d'un seul ensemble Q/K/V, utiliser `h` têtes en parallèle (ex. 4 ou 8), chacune apprenant à repérer un type de relation différent dans le texte. La concaténation de leurs sorties capture un contexte beaucoup plus riche, ce qui est le coeur de l'architecture Transformer originale.